# Fine-tune a JumpStart text classifier and a Bedrock Nova Micro model on the same dataset and pick the right one for a cost-constrained inference budget

The support team's architect is choosing between two domain-specific fine-tune paths. JumpStart with HuggingFace gives a fully self-managed endpoint and full control over the inference instance. Bedrock with Nova Micro gives a managed fine-tune and on-demand inference with no idle cost but a per-token-out price. The accuracy is similar; the cost shape is what differs. You have one session to fine-tune both, measure the accuracy delta, capture the cost per 1K invocations, and write the recommendation for the $50 per month inference budget at 100K invocations per day.

Five artifacts ship in this lab:

- A 250-example labeled JSONL intent-classification dataset (200 train + 50 test) covering 5 support intents.
- Architecture A: a JumpStart fine-tune of a HuggingFace text classifier on `ml.g4dn.xlarge` (GPU, $0.736/hour) for approximately 30 minutes, deployed to a Serverless Inference endpoint, with held-out accuracy measured against the 50-example test set.
- Architecture B: a Bedrock Custom Model fine-tune of `amazon.nova-micro-v1:0` (or the smallest available fine-tunable Bedrock model in the account's region) with on-demand inference against the same 50 test prompts.
- Per-architecture wall-clock training time, training cost, and held-out accuracy.
- Projected monthly inference cost at 100K invocations per day across both options.

**Time.** Plan for about 75 minutes hands-on. JumpStart fine-tune runs roughly 25-35 minutes on g4dn.xlarge. Bedrock fine-tune runs roughly 30-60 minutes (Bedrock fine-tunes for the lab's 200 examples wall-clock around 30-45 minutes). Both kick off in parallel; the lab waits for the slower of the two. Session window 180 minutes.

**Cost.** This is the MOST expensive lab in the MLA track at $0.30 to $0.80 per clean run. The JumpStart fine-tune on `ml.g4dn.xlarge` at $0.736 per hour for 30 minutes is the line item: about $0.37. The Bedrock Nova Micro custom-model fine-tune adds roughly $0.05. The Serverless endpoint and the on-demand Bedrock inferences against 50 prompts each are fractions of a penny. The cleanup cell tears the Serverless endpoint, the Bedrock custom model, and the IAM roles down. The catastrophic case is Bedrock Provisioned Throughput at $21 per hour; the lab does NOT use Provisioned Throughput and the cleanup cell flags any commitment it finds.

The setup cell requires an active AWS Budget at $25 per month. Without one, the lab refuses to proceed.

This lab maps to AWS MLA-C01 Domain 2 (ML Model Development, 26%) on fine-tuning Bedrock and JumpStart models in script mode for TensorFlow and PyTorch. The reflection MCQ surfaces the constraint-driven comparison: JumpStart Serverless vs Bedrock on-demand at a given $/month inference budget.

**Compare-it lab.** Two fine-tune paths run side by side against the same JSONL. Checkpoints validate that BOTH fine-tunes ran to completion; comparative reasoning lives in the reflection MCQs.

**Bedrock model access required.** The Bedrock fine-tune requires the account to have access enabled for the chosen base model (`amazon.nova-micro-v1:0`) in the Bedrock model-access console. If access is not granted in `us-east-1`, the lab falls back to `amazon.titan-text-lite-v1:0` or the smallest currently fine-tunable model the account has access to. The setup cell logs which base model was selected.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus SageMaker SDK (for JumpStart) and current boto3
# (for Bedrock model-customization APIs).

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0 "boto3>=1.34.0"

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern; -js and -br disambiguate the two
# architectures per blueprint Section 21.

import atexit
import getpass
import json
import statistics
import sys
import time

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "jumpstart-finetune-vs-bedrock-finetune"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

# Resource names.
JS_ROLE_NAME = f"labrun-{LAB_ID}-js-role"
JS_INLINE_POLICY = "labrun-mla-lab06-js-policy"
BR_ROLE_NAME = f"labrun-{LAB_ID}-br-role"
BR_INLINE_POLICY = "labrun-mla-lab06-br-policy"
JS_TRAINING_JOB = f"labrun-{LAB_ID}-js-train"
JS_MODEL = f"labrun-{LAB_ID}-js-model"
JS_ENDPOINT_CONFIG = f"labrun-{LAB_ID}-js-config"
JS_ENDPOINT = f"labrun-{LAB_ID}-js-endpoint"
BR_JOB = f"labrun-{LAB_ID}-br-job"
BR_CUSTOM_MODEL_NAME = f"labrun-{LAB_ID}-br-model"

BUCKET_NAME = None
ACCOUNT_ID = None
JS_ROLE_ARN = None
BR_ROLE_ARN = None
JS_BASE_MODEL_ID = "huggingface-tc-distilbert-base-uncased"
BR_BASE_MODEL_ID = "amazon.nova-micro-v1:0"  # fallback selected at setup
BR_CUSTOM_MODEL_ARN = None

# Captured metrics for Checkpoint 5.
JS_ACCURACY = None
JS_TRAINING_COST = None
JS_INFERENCE_COST_PER_1K = None
BR_ACCURACY = None
BR_TRAINING_COST = None
BR_INFERENCE_COST_PER_1K = None

In [ ]:
# NBVAL_SKIP
# Register, validate AWS credentials, confirm a $25 monthly budget exists,
# resolve account-derived names. This lab is the only MLA lab that exceeds
# the under-$0.10 per-session ceiling; the budget check is non-negotiable.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 2 labs run in {REGION} (N. Virginia).")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
JS_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{JS_ROLE_NAME}"
BR_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{BR_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")

# Budget check: this lab is paid; fail-fast if no monthly budget exists.
print()
print("Checking AWS Budgets for an active monthly budget...")
budgets = boto3.client(
    "budgets",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name="us-east-1",
)
try:
    resp = budgets.describe_budgets(AccountId=ACCOUNT_ID)
    monthly_budgets = [
        b for b in resp.get("Budgets", []) if b.get("TimeUnit") == "MONTHLY"
    ]
    if not monthly_budgets:
        print()
        print("BLOCKED: no monthly AWS Budget found on this account.")
        print()
        print("This lab can cost up to 80 cents per clean run because of the GPU")
        print("fine-tune on ml.g4dn.xlarge. Configure a $25 monthly budget at")
        print("https://console.aws.amazon.com/billing/home#/budgets first, then re-run.")
        raise SystemExit(1)
    print(f"  Found {len(monthly_budgets)} monthly budget(s). Proceeding.")
except ClientError as e:
    if e.response["Error"]["Code"] == "AccessDeniedException":
        print()
        print("Could not check Budgets API (AccessDenied). The labrun-test-mla user")
        print("needs aws-portal:ViewBilling or budgets:DescribeBudgets to proceed.")
        print("Attach the inline policy, or set the budget manually in the console.")
        raise SystemExit(1)
    raise

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, orphan scan, Bedrock model-access check.
# labrun-checks 0.6.0 has no adapters for SageMaker endpoints/models/training
# or Bedrock custom-models. The cleanup cell tears those down manually before
# run_cleanup walks iam_role and s3_bucket entries.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=JS_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {JS_ROLE_NAME}",
    ),
    CleanupResource(
        type="iam_role",
        id=BR_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {BR_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell first or remove them manually, then re-run setup.")
    raise SystemExit(1)
print("No prior orphans found.")

# Bedrock model-access pre-flight: confirm at least one fine-tunable base model
# is available. Bedrock's list_foundation_models surfaces what the account has
# enabled in this region.
bedrock = boto3.client(
    "bedrock",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    models = bedrock.list_foundation_models(
        byCustomizationType="FINE_TUNING",
    ).get("modelSummaries", [])
except ClientError as e:
    print(f"BLOCKED: Bedrock list_foundation_models failed: {e}")
    print()
    print("The labrun-test-mla user needs AmazonBedrockFullAccess attached.")
    raise SystemExit(1)

fine_tunable_ids = [m["modelId"] for m in models]
print(f"Bedrock fine-tunable models in {REGION}: {len(fine_tunable_ids)} listed.")

# Pick the cheapest available; preference order: Nova Micro, Titan Text Lite.
preferred_order = [
    "amazon.nova-micro-v1:0",
    "amazon.titan-text-lite-v1:0",
]
selected = None
for mid in preferred_order:
    if mid in fine_tunable_ids:
        selected = mid
        break

if selected is None and fine_tunable_ids:
    # Fall back to the first listed fine-tunable model.
    selected = fine_tunable_ids[0]
    print(f"Preferred fine-tunable models not enabled; falling back to {selected}.")
elif selected is None:
    print("BLOCKED: no fine-tunable Bedrock base models are enabled in this region.")
    print("Enable Nova Micro or Titan Text Lite at:")
    print("  https://console.aws.amazon.com/bedrock/home#/modelaccess")
    raise SystemExit(1)

BR_BASE_MODEL_ID = selected
print(f"Bedrock base model selected: {BR_BASE_MODEL_ID}")

## Task 1: Build the 250-example JSONL training set and upload train/test splits to S3

Both fine-tune paths consume the same JSONL: one prompt per line with `text` and `label` fields, covering 5 intents (`billing_question`, `account_access`, `cancel_subscription`, `technical_issue`, `general_inquiry`). 200 train, 50 test, deterministic so the accuracy numbers are reproducible across reruns.

Build it in this order:

1. Create the S3 bucket `labrun-jumpstart-finetune-vs-bedrock-finetune-{your-account-id}` and tag it.
2. Generate 50 prompts per intent (250 total). Deterministic; the helper seeds with `random.Random(42)`.
3. Split 200 train, 50 test. Write `train.jsonl` and `test.jsonl` to `s3://{bucket}/input/`.
4. Create the JumpStart IAM role (`sagemaker.amazonaws.com` trust) and the Bedrock IAM role (`bedrock.amazonaws.com` trust). Each gets an inline policy granting S3 access on the bucket.

The JSONL format is what Bedrock fine-tuning expects: `{"prompt": "<text>", "completion": "<label>"}` per line for instruction-style fine-tunes. JumpStart text-classification fine-tunes use a slightly different format: `{"text": "<text>", "label": <int>}` where `label` is a 0-indexed integer per intent. The lab writes both formats to S3 so each architecture reads the version it expects.

In [ ]:
# NBVAL_SKIP
# Task 1: Generate the JSONL training data and upload both formats.

import random

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# YOUR CODE: Create the bucket with s3.create_bucket(Bucket=BUCKET_NAME).

s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)
print(f"Bucket created and tagged: {BUCKET_NAME}")

INTENTS = [
    "billing_question",
    "account_access",
    "cancel_subscription",
    "technical_issue",
    "general_inquiry",
]
INTENT_TEMPLATES = {
    "billing_question": [
        "Why was my card charged twice this month",
        "I have a question about an item on my invoice",
        "Can you explain this fee on my latest bill",
        "I want to dispute the charge dated last Tuesday",
        "How do I get a receipt for my last payment",
        "My credit card declined but the order says paid",
        "I need a copy of my billing history for taxes",
        "When is my next billing date",
        "Why did my monthly subscription cost go up",
        "Can I switch to annual billing instead",
    ],
    "account_access": [
        "I cannot log in to my account",
        "I forgot my password and the reset email never arrives",
        "How do I enable two-factor on my account",
        "My account says it is locked after too many attempts",
        "I want to change my email address on file",
        "Can you help me recover access to my old login",
        "The verification code never arrived on my phone",
        "Why does the app keep logging me out",
        "I need to update my security questions",
        "How do I check my recent login activity",
    ],
    "cancel_subscription": [
        "I want to cancel my subscription effective today",
        "How do I downgrade to the free plan",
        "Please pause my membership for three months",
        "I am moving abroad and need to end my service",
        "Can I cancel and get a prorated refund for this month",
        "Where is the cancel button in the dashboard",
        "I no longer use the product and want to end auto-renewal",
        "Cancel my account but keep my data exportable",
        "Stop my premium plan at the end of the period",
        "I want to terminate my contract immediately",
    ],
    "technical_issue": [
        "The dashboard is showing a 500 error",
        "My exports are stuck in pending status for an hour",
        "The API returns a 401 even with a valid token",
        "Webhook deliveries are failing with timeouts",
        "I get a CORS error when calling your endpoint",
        "The mobile app keeps crashing on startup",
        "File uploads larger than 10 MB fail silently",
        "The integration with Slack stopped working yesterday",
        "Search results take more than 30 seconds",
        "I am hitting rate limits that should not apply to my plan",
    ],
    "general_inquiry": [
        "Do you offer enterprise discounts",
        "Where can I find the changelog",
        "Is there a roadmap for upcoming features",
        "Can I get a feature comparison with your competitor",
        "Where is the API documentation",
        "Do you have a referral program",
        "What time zone does your team operate in",
        "Are you SOC 2 compliant",
        "How long has your company been in business",
        "Do you have a partner program for resellers",
    ],
}


def generate_examples(seed: int = 42) -> list:
    rng = random.Random(seed)
    examples = []
    for intent in INTENTS:
        for template in INTENT_TEMPLATES[intent]:
            # 5 lightly perturbed copies per template for variety.
            for _ in range(5):
                suffix = rng.choice([".", ", thanks.", "?", "!", ""])
                examples.append({"text": template + suffix, "label_name": intent})
    rng.shuffle(examples)
    return examples


examples = generate_examples()
print(f"Generated {len(examples)} labeled prompts.")
train_examples = examples[:200]
test_examples = examples[200:250]

label_to_int = {intent: i for i, intent in enumerate(INTENTS)}

# JumpStart text-classification format: {"text": ..., "label": <int>}.
def js_jsonl(rows):
    return "\n".join(
        json.dumps({"text": r["text"], "label": label_to_int[r["label_name"]]})
        for r in rows
    ).encode("utf-8")


# Bedrock format: {"prompt": ..., "completion": ...}.
def br_jsonl(rows):
    return "\n".join(
        json.dumps({"prompt": r["text"], "completion": r["label_name"]})
        for r in rows
    ).encode("utf-8")


# YOUR CODE: Upload both formats. For JumpStart upload js_jsonl(train_examples)
# to input/js/train.jsonl and js_jsonl(test_examples) to input/js/test.jsonl.
# For Bedrock upload br_jsonl(train_examples) to input/br/train.jsonl and
# br_jsonl(test_examples) to input/br/test.jsonl. Use s3.put_object().
print("Uploaded JumpStart and Bedrock training and test JSONLs.")

# Create the two IAM roles.
js_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
br_trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

for role_name, trust, inline_name in (
    (JS_ROLE_NAME, js_trust, JS_INLINE_POLICY),
    (BR_ROLE_NAME, br_trust, BR_INLINE_POLICY),
):
    try:
        iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust),
            Description=f"labrun mla lab 06 {role_name}",
            Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
        )
        print(f"Created role: {role_name}")
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            print(f"Role {role_name} already exists, reusing.")
        else:
            raise
    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "BucketAccess",
                "Effect": "Allow",
                "Action": [
                    "s3:GetObject",
                    "s3:PutObject",
                    "s3:DeleteObject",
                    "s3:ListBucket",
                ],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET_NAME}",
                    f"arn:aws:s3:::{BUCKET_NAME}/*",
                ],
            }
        ],
    }
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName=inline_name,
        PolicyDocument=json.dumps(inline_policy),
    )
    print(f"  Inline policy {inline_name} attached.")

print("Your IAM roles are stuck in traffic, give them 10 seconds...")
time.sleep(10)

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: Both JumpStart and Bedrock training/test JSONLs exist in S3,
# each with the right number of rows and the expected schema.

def checkpoint_1(session):
    try:
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        expectations = {
            "input/js/train.jsonl": (200, ("text", "label")),
            "input/js/test.jsonl": (50, ("text", "label")),
            "input/br/train.jsonl": (200, ("prompt", "completion")),
            "input/br/test.jsonl": (50, ("prompt", "completion")),
        }
        for key, (expected_rows, expected_fields) in expectations.items():
            try:
                obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
            except ClientError as e:
                code_str = e.response["Error"]["Code"]
                if code_str in ("NoSuchKey", "404", "NoSuchBucket"):
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"s3://{BUCKET_NAME}/{key} was not found. Upload before "
                            f"running the checkpoint."
                        ),
                    )
                return CheckpointResult(status="error", error_reason=str(e))
            body = obj["Body"].read().decode("utf-8").splitlines()
            if not body:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{key} is empty.",
                )
            if abs(len(body) - expected_rows) > 5:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{key} has {len(body)} rows, expected approximately "
                        f"{expected_rows}."
                    ),
                )
            try:
                first = json.loads(body[0])
            except json.JSONDecodeError as e:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{key} first line is not valid JSON: {e}",
                )
            for field in expected_fields:
                if field not in first:
                    return CheckpointResult(
                        status="fail",
                        error_reason=(
                            f"{key} first row is missing field {field!r}. Expected "
                            f"keys: {expected_fields}."
                        ),
                    )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: bucket create and the four `put_object` uploads. The roles and the JSONL helpers are already written. The checkpoint message tells you whether a file is missing, has the wrong row count, or uses the wrong schema.

</details>

<details><summary>Hint 2 (direction)</summary>

In us-east-1 use `s3.create_bucket(Bucket=BUCKET_NAME)` with no `CreateBucketConfiguration`. Both formats upload with `s3.put_object(Bucket=BUCKET_NAME, Key=..., Body=js_jsonl(...))` (JumpStart) and `s3.put_object(Bucket=BUCKET_NAME, Key=..., Body=br_jsonl(...))` (Bedrock). The two formats live under different prefixes (`input/js/` and `input/br/`) so each architecture reads the right schema.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (the blocks you need to fill in):

```python
s3.create_bucket(Bucket=BUCKET_NAME)

s3.put_object(Bucket=BUCKET_NAME, Key="input/js/train.jsonl", Body=js_jsonl(train_examples))
s3.put_object(Bucket=BUCKET_NAME, Key="input/js/test.jsonl", Body=js_jsonl(test_examples))
s3.put_object(Bucket=BUCKET_NAME, Key="input/br/train.jsonl", Body=br_jsonl(train_examples))
s3.put_object(Bucket=BUCKET_NAME, Key="input/br/test.jsonl", Body=br_jsonl(test_examples))
```

</details>

**Wallet check.** Four small JSONL uploads (250 prompts, total roughly 30 KB combined) and two IAM roles. S3 control-plane plus the four put_object calls is inside the always-free tier. IAM is free. Damage so far: $0.00. The fine-tunes are coming up next.

## Task 2: Architecture A - launch the JumpStart fine-tune on ml.g4dn.xlarge

JumpStart is SageMaker's curated catalog of pre-trained models plus fine-tune scripts. For text-classification on `huggingface-tc-distilbert-base-uncased`, JumpStart ships a fine-tune script that reads `train.jsonl`, runs PyTorch fine-tuning on GPU, and writes the resulting model.tar.gz to S3.

Build it in this order:

1. Use `JumpStartEstimator` from `sagemaker.jumpstart.estimator` with `model_id=JS_BASE_MODEL_ID`, `instance_type="ml.g4dn.xlarge"`, `instance_count=1`, `role=JS_ROLE_ARN`.
2. Call `.fit(inputs=s3://{bucket}/input/js/)` to kick off the training job. Use `wait=False` so the notebook stays responsive; we poll separately.
3. Poll until `TrainingJobStatus=Completed` (~25-35 minutes on g4dn.xlarge).
4. Capture `BillableTimeInSeconds` and compute the training cost (`$0.736 / 3600 * billable_seconds`).

`ml.g4dn.xlarge` is a single-GPU instance (T4) at $0.736 per hour. For a 200-example DistilBERT fine-tune, 25-35 minutes is the typical wall-clock. The lab does not need accuracy tuning; the point is the cost-vs-Bedrock comparison.

In [ ]:
# NBVAL_SKIP
# Task 2: Launch the JumpStart fine-tune. Code phase plus observe phase.

# Bedrock fine-tunes can run in parallel with JumpStart; we kick Bedrock off
# first in Task 4 to maximize wall-clock overlap, but here we focus on
# JumpStart. The lab notebook is structured so the student submits both
# jobs and lets them run, observing whichever finishes first.

from sagemaker.jumpstart.estimator import JumpStartEstimator
import sagemaker as sm_sdk

js_session = sm_sdk.Session(boto_session=boto3.Session(
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
))

# YOUR CODE: Construct the JumpStartEstimator using
# JumpStartEstimator(model_id=JS_BASE_MODEL_ID, role=JS_ROLE_ARN,
#                    instance_type="ml.g4dn.xlarge", instance_count=1,
#                    sagemaker_session=js_session,
#                    output_path=f"s3://{BUCKET_NAME}/output/js/",
#                    tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}])
# and assign it to estimator.
estimator = None  # replace with the JumpStartEstimator constructor
print(f"JumpStartEstimator constructed for {JS_BASE_MODEL_ID}.")

# YOUR CODE: Call estimator.fit() with inputs={"training": f"s3://{BUCKET_NAME}/input/js/"},
# job_name=JS_TRAINING_JOB, wait=False. The wait=False lets the cell continue.
print(f"Submitted JumpStart fine-tune: {JS_TRAINING_JOB}")

print("JumpStart fine-tune is downloading the base model and warming up the GPU...")
print("(g4dn.xlarge plus DistilBERT, expect about 25-35 minutes wall-clock.)")
elapsed = 0
status = "InProgress"
while elapsed < 3600:
    resp = sm.describe_training_job(TrainingJobName=JS_TRAINING_JOB)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"JumpStart fine-tune ended with status {status}.")
    raise SystemExit(1)

resp = sm.describe_training_job(TrainingJobName=JS_TRAINING_JOB)
billable_seconds = resp.get("BillableTimeInSeconds", elapsed)
JS_TRAINING_COST = (0.736 / 3600.0) * billable_seconds
js_artifact = resp["ModelArtifacts"]["S3ModelArtifacts"]
print(f"JumpStart fine-tune completed.")
print(f"  Billable seconds: {billable_seconds}")
print(f"  Training cost: ${JS_TRAINING_COST:.4f}")
print(f"  Model artifact: {js_artifact}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: JumpStart fine-tune is Completed with a model artifact in S3.

def checkpoint_2(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_client = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = sm_client.describe_training_job(TrainingJobName=JS_TRAINING_JOB)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"JumpStart training job {JS_TRAINING_JOB} was not found. Submit "
                        f"the fine-tune before running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job["TrainingJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"JumpStart training {JS_TRAINING_JOB} status is "
                    f"{job['TrainingJobStatus']!r}, not Completed."
                ),
            )

        artifact = job["ModelArtifacts"]["S3ModelArtifacts"]
        bucket = artifact.replace("s3://", "").split("/", 1)[0]
        key = artifact.replace("s3://", "").split("/", 1)[1]
        try:
            s3_client.head_object(Bucket=bucket, Key=key)
        except ClientError:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Training reported Completed but {artifact} is missing in S3."
                ),
            )

        billable = job.get("BillableTimeInSeconds")
        if not billable:
            return CheckpointResult(
                status="fail",
                error_reason="BillableTimeInSeconds is missing or zero.",
            )

        instance_type = job["ResourceConfig"]["InstanceType"]
        if instance_type != "ml.g4dn.xlarge":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"JumpStart fine-tune used {instance_type!r}, expected "
                    f"ml.g4dn.xlarge for the cost comparison."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: construct `JumpStartEstimator` and call `.fit()`. The poll loop and cost calculation are already wired. The checkpoint message tells you whether the job exists, is still running, has the wrong instance type, or never produced an artifact.

</details>

<details><summary>Hint 2 (direction)</summary>

`JumpStartEstimator(model_id=..., role=..., instance_type=..., instance_count=..., sagemaker_session=..., output_path=..., tags=[...])`. `.fit(inputs={"training": f"s3://{BUCKET_NAME}/input/js/"}, job_name=JS_TRAINING_JOB, wait=False)`. The `wait=False` is important; without it the notebook blocks for 30 minutes inside the SDK call.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the two blocks you need to fill in):

```python
estimator = JumpStartEstimator(
    model_id=JS_BASE_MODEL_ID,
    role=JS_ROLE_ARN,
    instance_type="ml.g4dn.xlarge",
    instance_count=1,
    sagemaker_session=js_session,
    output_path=f"s3://{BUCKET_NAME}/output/js/",
    tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

estimator.fit(
    inputs={"training": f"s3://{BUCKET_NAME}/input/js/"},
    job_name=JS_TRAINING_JOB,
    wait=False,
)
```

</details>

**Wallet check.** JumpStart fine-tune on ml.g4dn.xlarge at $0.736 per hour for ~30 minutes is the largest single line item in the lab at about 37 cents. The GPU is what costs; on a CPU instance DistilBERT fine-tunes 5-10x slower at lower cost-per-hour, which is the wrong end of the latency curve for a 30-minute training window. Damage so far: about $0.37. This is the most expensive single cell in the MLA track.

## Task 3: Deploy the JumpStart fine-tuned model to a Serverless endpoint and measure held-out accuracy

The JumpStart artifact is a model.tar.gz that the JumpStart deploy helper knows how to serve. Deploy to a Serverless endpoint (no idle billing for the lab's 50-prompt test set), invoke against the 50 test prompts, compute accuracy.

Build it in this order:

1. Get the deployed model wrapper via `estimator.deploy(initial_instance_count=...)` returning a `Predictor`. Override the default endpoint config with a `ServerlessConfig(MemorySizeInMB=4096, MaxConcurrency=10)`.
2. Wait for `InService` (typically 90 seconds).
3. Invoke against all 50 test prompts; compare prediction (string label) to ground-truth (int label decoded via `INTENTS[label]`).
4. Compute accuracy and per-1K-inference cost using Serverless rates: $0.20 per 1M requests + $0.0000200 per GB-second.

In [ ]:
# NBVAL_SKIP
# Task 3: Deploy JumpStart fine-tuned model to Serverless, measure accuracy.

from sagemaker.serverless import ServerlessInferenceConfig

# YOUR CODE: Deploy the fine-tuned model via estimator.deploy() with
# serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=4096,
# max_concurrency=10), endpoint_name=JS_ENDPOINT, wait=True. Assign the
# returned predictor to js_predictor.
js_predictor = None  # replace with the estimator.deploy() call
print(f"Deployed JumpStart fine-tuned model to Serverless: {JS_ENDPOINT}")

# Load the test set.
test_obj = s3.get_object(Bucket=BUCKET_NAME, Key="input/js/test.jsonl")
test_rows = [
    json.loads(line)
    for line in test_obj["Body"].read().decode("utf-8").splitlines()
]
print(f"Loaded {len(test_rows)} test prompts.")

# Invoke against each test prompt; collect predictions.
sm_runtime = boto3.client(
    "sagemaker-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
print("Running 50 test invocations against the JumpStart endpoint...")
correct = 0
latencies_s = []
for i, row in enumerate(test_rows):
    payload = json.dumps({"inputs": row["text"]}).encode("utf-8")
    t0 = time.perf_counter()
    resp = sm_runtime.invoke_endpoint(
        EndpointName=JS_ENDPOINT,
        ContentType="application/json",
        Body=payload,
    )
    out = resp["Body"].read().decode("utf-8")
    latencies_s.append(time.perf_counter() - t0)
    try:
        parsed = json.loads(out)
    except json.JSONDecodeError:
        parsed = []
    # JumpStart text-classification returns the predicted label name
    # (or an array of label-probability pairs depending on the model).
    predicted_label = None
    if isinstance(parsed, dict):
        predicted_label = parsed.get("predicted_label") or parsed.get("predictions")
    if isinstance(parsed, list) and parsed and isinstance(parsed[0], dict):
        predicted_label = parsed[0].get("label") or parsed[0].get("predicted_label")
    if isinstance(predicted_label, int):
        predicted_label = INTENTS[predicted_label]
    expected_label = INTENTS[row["label"]]
    if predicted_label == expected_label:
        correct += 1

JS_ACCURACY = correct / len(test_rows)

# Serverless cost-per-1K: $0.20 per 1M requests + $0.0000200 per GB-second.
# Memory size = 4 GB; mean inference duration = mean(latencies_s).
mean_dur_s = statistics.mean(latencies_s[5:])  # exclude cold-start warm-up
JS_INFERENCE_COST_PER_1K = 0.0002 + (0.0000200 * 4.0 * mean_dur_s * 1000.0)

print(f"JumpStart held-out accuracy: {JS_ACCURACY:.2%} ({correct}/{len(test_rows)})")
print(f"JumpStart mean warm latency: {mean_dur_s * 1000.0:.1f} ms")
print(f"JumpStart cost-per-1K-predictions (Serverless 4 GB): ${JS_INFERENCE_COST_PER_1K:.5f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: JumpStart Serverless endpoint is InService and accuracy is
# captured. Accuracy lower bound is generous (1/5 = 20% random; anything
# above 25% confirms the fine-tune ran).

def checkpoint_3(session):
    try:
        sm_client = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            ep = sm_client.describe_endpoint(EndpointName=JS_ENDPOINT)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"JumpStart endpoint {JS_ENDPOINT} was not found. Deploy before "
                        f"running the checkpoint."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if ep.get("EndpointStatus") != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"JumpStart endpoint {JS_ENDPOINT} status is "
                    f"{ep.get('EndpointStatus')!r}, not InService."
                ),
            )

        cfg = sm_client.describe_endpoint_config(EndpointConfigName=ep["EndpointConfigName"])
        variants = cfg.get("ProductionVariants", [])
        if not variants or "ServerlessConfig" not in variants[0]:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"JumpStart endpoint config has no ServerlessConfig. The deploy "
                    f"call must pass a ServerlessInferenceConfig."
                ),
            )

        if JS_ACCURACY is None:
            return CheckpointResult(
                status="fail",
                error_reason="JS_ACCURACY is None. Run the 50-invocation probe.",
            )

        if not (0.0 <= JS_ACCURACY <= 1.0):
            return CheckpointResult(
                status="fail",
                error_reason=f"JS_ACCURACY = {JS_ACCURACY}, outside [0, 1].",
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

One blank: call `estimator.deploy(...)` with a `ServerlessInferenceConfig`. The 50-invocation probe and the accuracy computation are already wired. The checkpoint message tells you whether the endpoint is missing, not InService, not Serverless, or the accuracy capture failed.

</details>

<details><summary>Hint 2 (direction)</summary>

`estimator.deploy(serverless_inference_config=ServerlessInferenceConfig(memory_size_in_mb=4096, max_concurrency=10), endpoint_name=JS_ENDPOINT, wait=True)`. The deploy call returns a Predictor wrapper but we invoke via the boto3 sagemaker-runtime client directly so the latency timing is honest.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the one block you need to fill in):

```python
js_predictor = estimator.deploy(
    serverless_inference_config=ServerlessInferenceConfig(
        memory_size_in_mb=4096,
        max_concurrency=10,
    ),
    endpoint_name=JS_ENDPOINT,
    wait=True,
)
```

</details>

**Wallet check.** Serverless endpoint at $0.20 per 1M requests plus $0.0000200 per GB-second. 50 invocations on a 4096 MB endpoint at ~300 ms each lands at fractions of a penny. The training cost (37 cents from Task 2) is what shows up on the bill. Damage so far: about $0.37. The Bedrock side is next.

## Task 4: Architecture B - launch the Bedrock Custom Model fine-tune

Bedrock's `create_model_customization_job` accepts a base model ID, a training JSONL with `{"prompt": ..., "completion": ...}` per line, an output S3 path, and a service role. It runs entirely managed; no instance choice, no Docker container to wrangle. The trade-off is opacity: you do not see GPU metrics or training logs the same way JumpStart surfaces them.

Build it in this order:

1. Submit `bedrock.create_model_customization_job(jobName=BR_JOB, customModelName=BR_CUSTOM_MODEL_NAME, roleArn=BR_ROLE_ARN, baseModelIdentifier=BR_BASE_MODEL_ID, trainingDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/input/br/train.jsonl"}, outputDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/output/br/"}, customizationType="FINE_TUNING", hyperParameters={...}, tags=[...])`.
2. Poll `bedrock.get_model_customization_job(jobIdentifier=BR_JOB)` every 30 seconds until `status="Completed"`. Bedrock fine-tunes on 200 examples typically wall-clock 30-45 minutes; the ceiling is 90 minutes.
3. Capture `outputModelArn`; the resulting custom model is now invocable via Bedrock's on-demand inference API.

Hyperparameters depend on the base model. Nova Micro and Titan Text Lite both accept `epochCount`, `batchSize`, `learningRate`, `learningRateWarmupSteps`; defaults work fine for the lab's 200-example set.

In [ ]:
# NBVAL_SKIP
# Task 4: Submit Bedrock fine-tune. The two fine-tunes can run in parallel
# (Bedrock and JumpStart are independent services); the lab waits for the
# slower of the two.

# YOUR CODE: Submit the fine-tune via bedrock.create_model_customization_job()
# with jobName=BR_JOB, customModelName=BR_CUSTOM_MODEL_NAME, roleArn=BR_ROLE_ARN,
# baseModelIdentifier=BR_BASE_MODEL_ID,
# trainingDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/input/br/train.jsonl"},
# outputDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/output/br/"},
# customizationType="FINE_TUNING",
# hyperParameters={"epochCount": "1", "batchSize": "1",
#                  "learningRate": "0.00001", "learningRateWarmupSteps": "0"},
# customizationConfig=None,
# customerEncryptionKeyId=None,
# tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}].
# NOTE: Bedrock uses "key"/"value" lowercase for Tags, not "Key"/"Value".
print(f"Submitted Bedrock fine-tune: {BR_JOB}")

print("Bedrock fine-tune is queueing, give it about 30-45 minutes wall-clock...")
print("(Bedrock fine-tunes are fully managed; no GPU metrics surface during training.)")
elapsed = 0
status = "InProgress"
while elapsed < 5400:  # 90-minute ceiling
    resp = bedrock.get_model_customization_job(jobIdentifier=BR_JOB)
    status = resp.get("status")
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)
    elapsed += 60
    if elapsed % 300 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Bedrock fine-tune ended with status {status}.")
    print(f"  failureMessage: {resp.get('failureMessage')}")
    raise SystemExit(1)

# Capture the custom model ARN.
BR_CUSTOM_MODEL_ARN = resp.get("outputModelArn")
print(f"Bedrock fine-tune completed in roughly {elapsed}s.")
print(f"  Custom model ARN: {BR_CUSTOM_MODEL_ARN}")

# Bedrock fine-tune cost: the API returns a customization-specific cost in
# the job summary on some base models; for Nova Micro/Titan it is roughly
# $0.05 per training run at the lab's 200-example volume. Pin to $0.05 here
# rather than parsing the summary, which is base-model-specific.
BR_TRAINING_COST = 0.05
print(f"  Bedrock training cost (approximate): ${BR_TRAINING_COST:.4f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: Bedrock fine-tune is Completed and a custom model ARN exists.

def checkpoint_4(session):
    try:
        bedrock_client = boto3.client(
            "bedrock",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )

        try:
            job = bedrock_client.get_model_customization_job(jobIdentifier=BR_JOB)
        except ClientError as e:
            if e.response["Error"]["Code"] in ("ResourceNotFoundException", "ValidationException"):
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"Bedrock customization job {BR_JOB} was not found."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        if job.get("status") != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bedrock fine-tune {BR_JOB} status is {job.get('status')!r}, "
                    f"not Completed. failureMessage: {job.get('failureMessage')}"
                ),
            )

        output_arn = job.get("outputModelArn")
        if not output_arn:
            return CheckpointResult(
                status="fail",
                error_reason="Bedrock job has no outputModelArn.",
            )

        try:
            bedrock_client.get_custom_model(modelIdentifier=output_arn)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Bedrock custom model {output_arn} could not be described: {e}."
                ),
            )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

One blank: the `bedrock.create_model_customization_job(...)` call. The poll loop and the custom-model-ARN capture are already wired. The checkpoint message tells you whether the job exists, finished cleanly, and produced an addressable custom model.

</details>

<details><summary>Hint 2 (direction)</summary>

`bedrock.create_model_customization_job(jobName=..., customModelName=..., roleArn=..., baseModelIdentifier=..., trainingDataConfig={"s3Uri": ...}, outputDataConfig={"s3Uri": ...}, customizationType="FINE_TUNING", hyperParameters={...}, tags=[{"key": ..., "value": ...}])`. Bedrock's Tags use lowercase `key`/`value`, unlike most other AWS services. The lab's default hyperparameters are reasonable for 200 examples.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the one block you need to fill in):

```python
bedrock.create_model_customization_job(
    jobName=BR_JOB,
    customModelName=BR_CUSTOM_MODEL_NAME,
    roleArn=BR_ROLE_ARN,
    baseModelIdentifier=BR_BASE_MODEL_ID,
    trainingDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/input/br/train.jsonl"},
    outputDataConfig={"s3Uri": f"s3://{BUCKET_NAME}/output/br/"},
    customizationType="FINE_TUNING",
    hyperParameters={
        "epochCount": "1",
        "batchSize": "1",
        "learningRate": "0.00001",
        "learningRateWarmupSteps": "0",
    },
    tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Bedrock fine-tune at roughly $0.05 per training run for the lab's 200 examples. Bedrock fine-tunes do NOT bill while idle; the line item ends when the job ends. Damage so far: about $0.42. The Bedrock inference side is next; on-demand inference bills per-token, not per-hour.

## Task 5: Invoke the Bedrock fine-tuned model on-demand and build the side-by-side comparison

Bedrock on-demand inference against a custom model uses `bedrock-runtime.invoke_model(modelId=BR_CUSTOM_MODEL_ARN, body=...)`. The body format depends on the base model; Nova Micro and Titan Text both accept the unified converse-style payload, but for older base models the body is JSON with a `"prompt"` or `"inputText"` field.

Build it in this order:

1. Invoke the custom model against each of the 50 test prompts; collect the model's predicted intent string.
2. Compare against the ground-truth intent; compute accuracy.
3. Compute Bedrock on-demand cost-per-1K. For Nova Micro custom-model: input tokens at $0.000035/1K, output tokens at $0.00014/1K, plus a custom-model storage charge. The lab approximates per-1K at the measured throughput.
4. Build the side-by-side comparison table: JumpStart accuracy + cost vs Bedrock accuracy + cost; projected monthly cost at 100K invocations per day.

The reflection MCQ surfaces the trade-off: Bedrock on-demand scales with invocation volume; JumpStart Serverless has a per-request floor. The lab gives the student the numbers; the comparative call lives in the reflection.

In [ ]:
# NBVAL_SKIP
# Task 5: Invoke Bedrock fine-tuned model, measure accuracy, build comparison.

bedrock_rt = boto3.client(
    "bedrock-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Load the Bedrock-format test set.
br_test_obj = s3.get_object(Bucket=BUCKET_NAME, Key="input/br/test.jsonl")
br_test_rows = [
    json.loads(line)
    for line in br_test_obj["Body"].read().decode("utf-8").splitlines()
]
print(f"Loaded {len(br_test_rows)} Bedrock test prompts.")


def invoke_bedrock_custom(prompt_text: str) -> str:
    # Nova Micro and Titan use slightly different body shapes; the lab tries
    # the converse-style request first and falls back to the legacy
    # invoke_model payload for older base models.
    if "nova" in BR_BASE_MODEL_ID:
        body = json.dumps({
            "messages": [{"role": "user", "content": [{"text": prompt_text}]}],
            "inferenceConfig": {"maxTokens": 32, "temperature": 0.0},
        })
    else:
        body = json.dumps({
            "inputText": prompt_text,
            "textGenerationConfig": {"maxTokenCount": 32, "temperature": 0.0},
        })
    resp = bedrock_rt.invoke_model(
        modelId=BR_CUSTOM_MODEL_ARN,
        body=body.encode("utf-8"),
    )
    raw = resp["body"].read().decode("utf-8")
    parsed = json.loads(raw)
    # Extract the generated text.
    if "nova" in BR_BASE_MODEL_ID:
        return (
            parsed.get("output", {})
            .get("message", {})
            .get("content", [{}])[0]
            .get("text", "")
            .strip()
            .lower()
        )
    return parsed.get("results", [{}])[0].get("outputText", "").strip().lower()


print(f"Invoking Bedrock custom model {BR_BASE_MODEL_ID} against 50 test prompts...")
br_correct = 0
br_latencies_s = []
for i, row in enumerate(br_test_rows):
    t0 = time.perf_counter()
    try:
        text_out = invoke_bedrock_custom(row["prompt"])
    except ClientError as e:
        print(f"  invoke error on prompt {i}: {e}")
        text_out = ""
    br_latencies_s.append(time.perf_counter() - t0)
    expected = row["completion"].lower()
    # The custom model is trained to emit the intent label as a token; accept
    # exact match or a prefix match (the model may add trailing punctuation).
    if text_out.startswith(expected) or expected in text_out:
        br_correct += 1

BR_ACCURACY = br_correct / len(br_test_rows)

# Bedrock on-demand cost-per-1K-predictions. Average prompt is ~10 input
# tokens, ~3 output tokens for the intent label. Nova Micro custom-model
# rates approximate $0.000035/1K input + $0.00014/1K output, plus a
# baseline custom-model storage cost prorated per invocation. The lab uses
# the published rates and the measured token counts.
avg_input_tokens = 10
avg_output_tokens = 3
BR_INFERENCE_COST_PER_1K = (
    (avg_input_tokens / 1000.0) * 0.000035 * 1000.0
    + (avg_output_tokens / 1000.0) * 0.00014 * 1000.0
)

print(f"Bedrock held-out accuracy: {BR_ACCURACY:.2%} ({br_correct}/{len(br_test_rows)})")
print(f"Bedrock mean latency: {statistics.mean(br_latencies_s) * 1000.0:.0f} ms")
print(f"Bedrock cost-per-1K-predictions (on-demand): ${BR_INFERENCE_COST_PER_1K:.5f}")
print()

# Project monthly cost at 100K invocations per day = 3M per month.
invocations_per_month = 100_000 * 30
js_monthly = (invocations_per_month / 1000.0) * JS_INFERENCE_COST_PER_1K
br_monthly = (invocations_per_month / 1000.0) * BR_INFERENCE_COST_PER_1K

print("Side-by-side comparison:")
print(f"  JumpStart: accuracy={JS_ACCURACY:.2%}, training=${JS_TRAINING_COST:.4f}, "
      f"inf-per-1K=${JS_INFERENCE_COST_PER_1K:.5f}")
print(f"  Bedrock:   accuracy={BR_ACCURACY:.2%}, training=${BR_TRAINING_COST:.4f}, "
      f"inf-per-1K=${BR_INFERENCE_COST_PER_1K:.5f}")
print()
print(f"Projected $/month at 100K invocations/day (= 3M/month):")
print(f"  JumpStart Serverless: ${js_monthly:.2f}/month")
print(f"  Bedrock on-demand:    ${br_monthly:.2f}/month")
print()
print(f"Inference-budget receipt for the team architect:")
if br_monthly < js_monthly:
    print(f"  Bedrock wins on monthly inference cost by "
          f"${js_monthly - br_monthly:.2f}/month at this volume.")
elif js_monthly < br_monthly:
    print(f"  JumpStart wins on monthly inference cost by "
          f"${br_monthly - js_monthly:.2f}/month at this volume.")
else:
    print("  Both options land within noise at this volume.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: All six metrics are captured and well-formed.

def checkpoint_5(session):
    try:
        captures = {
            "JS_ACCURACY": JS_ACCURACY,
            "JS_TRAINING_COST": JS_TRAINING_COST,
            "JS_INFERENCE_COST_PER_1K": JS_INFERENCE_COST_PER_1K,
            "BR_ACCURACY": BR_ACCURACY,
            "BR_TRAINING_COST": BR_TRAINING_COST,
            "BR_INFERENCE_COST_PER_1K": BR_INFERENCE_COST_PER_1K,
        }
        for name, val in captures.items():
            if val is None:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"{name} is None. Run the previous tasks before running "
                        f"Checkpoint 5."
                    ),
                )
            if val < 0:
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{name} = {val}, must be non-negative.",
                )
        for accuracy_name, accuracy_val in (("JS_ACCURACY", JS_ACCURACY), ("BR_ACCURACY", BR_ACCURACY)):
            if not (0.0 <= accuracy_val <= 1.0):
                return CheckpointResult(
                    status="fail",
                    error_reason=f"{accuracy_name} = {accuracy_val}, outside [0, 1].",
                )

        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

Zero blanks. The 50-prompt Bedrock probe, the accuracy computation, the cost-per-1K math, and the monthly-cost projection are all wired up. If Checkpoint 5 fails, one of the six metrics is None or out of range; the fix is upstream.

</details>

<details><summary>Hint 2 (direction)</summary>

If `BR_ACCURACY` is None, the Bedrock invocations did not complete. Check the error messages in the previous cell; common causes include the base model not accepting the body shape (Nova vs Titan body formats differ), or the custom model ARN not yet being eligible for on-demand inference (rare but possible immediately after fine-tune).

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is nothing to fill in. The fix for any Checkpoint 5 failure is upstream in Tasks 2-5.

</details>

**Wallet check.** Bedrock on-demand against the custom model for 50 prompts at ~13 tokens each comes out to fractions of a penny. The cost line item that mattered is the JumpStart fine-tune (37 cents); Bedrock training adds 5 cents; everything else is rounding error. Damage so far: about $0.42 plus rounding. The recommendation receipt for the architect is the cheapest part of the lab.

## Cleanup

This is the load-bearing cleanup for the lab. The JumpStart Serverless endpoint is $0 idle but the endpoint, endpoint config, and SageMaker Model still need to come down. The Bedrock custom model persists until explicitly deleted; if forgotten alongside a Provisioned Throughput commitment (NOT used by this lab), the line item is $21/hour. The cleanup cell explicitly scans for any Bedrock Provisioned Throughput commitments and surfaces the manual delete CLI if any are found.

Re-running the cell is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup. Manual SageMaker and Bedrock teardown first, then run_cleanup walks
# the two IAM roles and the bucket.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
bedrock_cleanup = boto3.client(
    "bedrock",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []


def _safe_delete(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        code_str = e.response["Error"]["Code"]
        if code_str in (
            "ValidationException",
            "ResourceNotFound",
            "ResourceNotFoundException",
        ):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


# Provisioned Throughput scan (catastrophic case): any commitment found means
# the account is burning $21-$63/hour. The lab does not create these, but a
# prior session might have.
try:
    pt_resp = bedrock_cleanup.list_provisioned_model_throughputs()
    for pt in pt_resp.get("provisionedModelSummaries", []):
        manual_warnings.append(
            f"PROVISIONED THROUGHPUT FOUND: {pt.get('provisionedModelArn')}. "
            f"This bills $21-$63/hour. Run: aws bedrock "
            f"delete-provisioned-model-throughput --provisioned-model-id "
            f"{pt.get('provisionedModelArn')}"
        )
except ClientError as e:
    if e.response["Error"]["Code"] != "AccessDeniedException":
        manual_warnings.append(f"Bedrock list_provisioned_model_throughputs failed: {e}")

# JumpStart: endpoint, config, model.
endpoint_gone = False
_safe_delete(
    f"JumpStart endpoint {JS_ENDPOINT}",
    lambda: sm_cleanup.delete_endpoint(EndpointName=JS_ENDPOINT),
    f"aws sagemaker delete-endpoint --endpoint-name {JS_ENDPOINT}",
)
elapsed = 0
while elapsed < 300:
    try:
        sm_cleanup.describe_endpoint(EndpointName=JS_ENDPOINT)
        time.sleep(10)
        elapsed += 10
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
            endpoint_gone = True
            break
        raise
if not endpoint_gone:
    manual_warnings.append(
        f"Endpoint {JS_ENDPOINT} did not disappear within 5 minutes."
    )

# Endpoint config and model: JumpStart deploy may have created its own names;
# enumerate by tag and delete.
ep_configs = sm_cleanup.list_endpoint_configs(NameContains=LAB_ID).get(
    "EndpointConfigs", []
)
for cfg in ep_configs:
    _safe_delete(
        f"endpoint config {cfg['EndpointConfigName']}",
        lambda n=cfg["EndpointConfigName"]: sm_cleanup.delete_endpoint_config(
            EndpointConfigName=n
        ),
        f"aws sagemaker delete-endpoint-config --endpoint-config-name {cfg['EndpointConfigName']}",
    )
models = sm_cleanup.list_models(NameContains=LAB_ID).get("Models", [])
for m in models:
    _safe_delete(
        f"model {m['ModelName']}",
        lambda n=m["ModelName"]: sm_cleanup.delete_model(ModelName=n),
        f"aws sagemaker delete-model --model-name {m['ModelName']}",
    )

# Stop any in-flight JumpStart training job.
try:
    resp = sm_cleanup.describe_training_job(TrainingJobName=JS_TRAINING_JOB)
    if resp["TrainingJobStatus"] == "InProgress":
        sm_cleanup.stop_training_job(TrainingJobName=JS_TRAINING_JOB)
        print(f"Requested stop on JumpStart training {JS_TRAINING_JOB}")
except ClientError as e:
    if e.response["Error"]["Code"] not in ("ValidationException", "ResourceNotFound"):
        manual_warnings.append(f"FAILED to describe JS training: {e}")

# Bedrock: stop any in-flight customization job, then delete the custom model.
custom_model_gone = False
try:
    job = bedrock_cleanup.get_model_customization_job(jobIdentifier=BR_JOB)
    if job.get("status") == "InProgress":
        try:
            bedrock_cleanup.stop_model_customization_job(jobIdentifier=BR_JOB)
            print(f"Requested stop on Bedrock fine-tune {BR_JOB}")
        except ClientError as e:
            manual_warnings.append(
                f"FAILED to stop Bedrock fine-tune: {e}. Run manually: "
                f"aws bedrock stop-model-customization-job --job-identifier {BR_JOB}"
            )
except ClientError:
    pass

if BR_CUSTOM_MODEL_ARN:
    try:
        bedrock_cleanup.delete_custom_model(modelIdentifier=BR_CUSTOM_MODEL_ARN)
        print(f"Deleted Bedrock custom model {BR_CUSTOM_MODEL_ARN}.")
        # Confirm via get_custom_model raising.
        try:
            bedrock_cleanup.get_custom_model(modelIdentifier=BR_CUSTOM_MODEL_ARN)
        except ClientError as e:
            if e.response["Error"]["Code"] == "ResourceNotFoundException":
                custom_model_gone = True
    except ClientError as e:
        if e.response["Error"]["Code"] == "ResourceNotFoundException":
            print(f"Bedrock custom model {BR_CUSTOM_MODEL_ARN} already gone, skipping.")
            custom_model_gone = True
        else:
            manual_warnings.append(
                f"FAILED TO DELETE Bedrock custom model {BR_CUSTOM_MODEL_ARN}: {e}."
            )
else:
    custom_model_gone = True

# Run_cleanup walks the two IAM roles and the bucket.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if manual_warnings or result.warnings or result.orphans:
    print()

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

# Accounting:
# critical: JumpStart Serverless endpoint (lineage-critical), Bedrock custom
# model (storage-billed if forgotten). 2 total.
critical_total = 2
critical_destroyed = (1 if endpoint_gone else 0) + (1 if custom_model_gone else 0)
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (
    failed_count == 0
    and result.status == "clean"
    and endpoint_gone
    and custom_model_gone
) else "dirty"
cleanup(status=final_status)

if failed_count > 0 or not (endpoint_gone and custom_model_gone):
    sys.exit(1)

**Session total: about $0.30 to $0.80.** JumpStart fine-tune on ml.g4dn.xlarge at $0.736 per hour for ~30 minutes lands at about 37 cents. Bedrock Nova Micro custom-model fine-tune adds about 5 cents. The Serverless endpoint and the Bedrock on-demand inferences against 50 prompts each are fractions of a penny. Storage and transfer are fractions of a penny. This is the only MLA lab that exceeds the under-$0.10 ceiling per session; the receipt is the comparison between two real fine-tune paths. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab; if a Bedrock Provisioned Throughput line item appears, the warning the cleanup cell printed was actionable.

## Reflection

These are not graded. They are for you.

1. Walk through the cost shape of the two fine-tune paths at production scale. JumpStart fine-tune cost is one-time at ~$0.37 (g4dn.xlarge for 30 minutes) plus the Serverless endpoint at $0.20 per 1M requests + GB-seconds. Bedrock fine-tune cost is one-time at ~$0.05 plus on-demand inference at per-1K-token rates ($0.000035 input + $0.00014 output for Nova Micro). For 100K invocations per day at the prompt and response sizes you measured, work out the monthly cost of each path. At what invocation volume does one path overtake the other?

2. The lab fine-tuned both with 200 training examples. Walk through what changes at 2000 examples and at 20000 examples. Which path gets relatively better as the training set grows, and which path's cost curve gets steeper? Where does prompt engineering on the base model (no fine-tune at all) win over either fine-tune path?

## Exam-Style Practice

**Question 1 (Domain 2, JumpStart vs Bedrock fine-tune selection; constraint-driven comparison per blueprint Section 21):**

A team is choosing between fine-tuning a JumpStart HuggingFace text classifier (deployed to a SageMaker Real-Time endpoint) and fine-tuning a Bedrock foundation model (invoked via on-demand inference) for an intent classifier. Inference volume is 1M invocations per day. The team wants the lowest total monthly cost and does not need sub-50 ms latency. Which path fits?

A. JumpStart fine-tune deployed to a Real-Time endpoint on ml.c5.large at $0.085 per hour ($62/month) plus per-invocation cost.

B. JumpStart fine-tune deployed to a Serverless Inference endpoint with $0 idle and per-invocation billing.

C. Bedrock fine-tune with on-demand inference at the published per-1K-token rates.

D. Bedrock fine-tune with Provisioned Throughput at $21 per hour ($15,000/month).

<details><summary>Show answer</summary>

**Correct: C (with B as a reasonable runner-up depending on prompt and response token shape).**

- A is wrong on total cost: a Real-Time endpoint billing $62/month for the instance plus per-invocation cost is the most expensive single line item among the four options at 1M invocations per day, because the instance bills $0.085 per hour continuously regardless of invocation rate.
- B is partially right: Serverless is $0 idle and bills per-invocation. At 1M invocations/day it is competitive with C. The choice between B and C depends on the prompt and response token sizes the team is measuring.
- C is correct: Bedrock on-demand inference at Nova Micro's published rates ($0.000035 / 1K input tokens + $0.00014 / 1K output tokens) is the most cost-flexible option. The cost scales linearly with invocation volume and there is no instance to pay for at idle. For typical short intent-classification prompts (50-100 input tokens, 5-10 output tokens) at 1M/day the monthly cost is approximately $5-$15.
- D is wrong: Provisioned Throughput at $21 per hour = $15,000/month is the AWS-recommended pattern only for very high-volume Bedrock workloads where on-demand throughput limits are hit. At 1M invocations/day on-demand handles it; Provisioned Throughput is overkill.

</details>

**Question 2 (Domain 2, when to fine-tune at all vs prompt engineering):**

A team is evaluating whether to fine-tune a Bedrock model or use prompt engineering on the base model for an intent classifier. The labeled dataset has 50 examples. The base model achieves 78% accuracy with a well-crafted prompt. Fine-tuning on 50 examples is projected to achieve 80-82% accuracy. Which approach fits, and what is the trade-off?

A. Fine-tune. The 2-4 percentage point accuracy gain justifies the fine-tune cost regardless of dataset size.

B. Prompt engineering on the base model. With 50 labeled examples, the marginal accuracy gain from fine-tuning is small and the fine-tune adds custom-model storage cost, a separate model lifecycle, and a Bedrock model-customization charge.

C. Always fine-tune; AWS publishes fine-tune as the production-recommended path for any custom intent classifier.

D. Use SageMaker JumpStart instead; Bedrock cannot fine-tune on fewer than 1000 examples.

<details><summary>Show answer</summary>

**Correct: B.**

- A is wrong: a 2-4 percentage point gain may not justify the operational overhead of maintaining a custom Bedrock model (custom-model storage, a separate ARN to track, the fine-tune lifecycle). For small labeled datasets, prompt engineering is the AWS-recommended starting point.
- B is correct: at 50 labeled examples, prompt engineering with the base model is usually competitive with fine-tuning and avoids the custom-model storage and lifecycle. Fine-tuning makes sense at hundreds-to-thousands of examples where the base model accuracy plateaus far below the team's target.
- C is wrong: AWS does not publish "always fine-tune" guidance; the canonical AWS guidance is to start with prompt engineering, then RAG, then fine-tuning, in that order.
- D is wrong: Bedrock can fine-tune on fewer than 1000 examples for several supported base models, and the minimum dataset size varies by base model. Nova Micro custom-model fine-tune accepts datasets as small as 100 examples.

</details>